# Link Prediction — v4 (GNN avec DropEdge + décodeur symétrique)

## Récapitulatif des problèmes et solutions

| Problème | Cause | Fix v4 |
|---|---|---|
| Val AUC truquée (v2) | Features structurelles calculées avec arêtes val dans G | Split avant construction G |
| GNN overfit structurel | GCN propage sur arêtes train → `z_u` absorbe features de `v` | **DropEdge** pendant le training |
| Décodeur asymétrique | `[z_u | z_v]` dépend de l'ordre, double la capacité inutilement | **`[z_u * z_v | |z_u - z_v|]`** |
| Oversmoothing / trop de params | 3 couches GCN | **SAGEConv 2 couches** |
| Features brutes non normalisées | TF-IDF → grande variance entre dims | **Normalisation L2** avant GNN |

### Pourquoi DropEdge aide
À chaque forward, on masque aléatoirement ~40% des arêtes. Le GNN ne peut plus se fier
à une arête spécifique pour calculer l'embedding d'un nœud. Il est forcé d'apprendre
des représentations robustes basées sur le **voisinage global**, pas sur des arêtes précises.
Pendant l'inférence, toutes les arêtes sont utilisées (comme le MC Dropout).

### Pourquoi le décodeur symétrique
`[z_u | z_v]` : 2×64 = 128 entrées, **asymétrique** (prédire (u,v) ≠ prédire (v,u) avant MLP)
`[z_u * z_v | |z_u - z_v|]` : 2×64 = 128 entrées, **symétrique**, la moitié des paramètres effectifs,
capture mieux la similarité et la dissimilarité entre embeddings.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.utils import dropout_edge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement et split

In [2]:
train_full = pd.read_csv("data/train.txt", sep=" ", header=None)
train_full.columns = ["u", "v", "label"]

test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]

node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"})
node_features_raw = {
    int(row["node"]): row.drop("node").values.astype(np.float32)
    for _, row in node_info.iterrows()
}
feature_dim = len(next(iter(node_features_raw.values())))

# Split AVANT construction du graphe
train_df, val_df = train_test_split(
    train_full, test_size=0.2, stratify=train_full["label"], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Features : {feature_dim} dims")

Train : 8396 | Val : 2100 | Features : 932 dims


## 2. Graphe — arêtes positives du train uniquement

In [3]:
G = nx.Graph()
G.add_edges_from(train_df[train_df["label"] == 1][["u", "v"]].values)

# Ajouter tous les nœuds (isolés ou non) pour avoir des embeddings partout
for u, v in pd.concat([train_full[["u","v"]], test[["u","v"]]]).values:
    if not G.has_node(int(u)): G.add_node(int(u))
    if not G.has_node(int(v)): G.add_node(int(v))

degree     = dict(G.degree())
components = {n: c for c, comp in enumerate(nx.connected_components(G)) for n in comp}
feat_norms = {n: np.linalg.norm(f) + 1e-9 for n, f in node_features_raw.items()}

print(f"Nœuds : {G.number_of_nodes()} | Arêtes train positives : {G.number_of_edges()}")

Nœuds : 3597 | Arêtes train positives : 4198


## 3. Features structurelles de paires

In [4]:
def structural_features(u, v):
    deg_u = degree.get(u, 0)
    deg_v = degree.get(v, 0)
    if G.has_node(u) and G.has_node(v):
        nu, nv = set(G.neighbors(u)), set(G.neighbors(v))
        inter  = nu & nv
        union  = nu | nv
        cn     = len(inter)
        jacc   = cn / len(union) if union else 0.0
        aa     = sum(1.0 / np.log(degree[w] + 1e-9) for w in inter if degree.get(w, 0) > 1)
        pa     = deg_u * deg_v
        sc     = float(components.get(u, -1) == components.get(v, -2))
    else:
        cn, jacc, aa, pa, sc = 0, 0.0, 0.0, 0, 0.0
    cosine = 0.0
    if u in node_features_raw and v in node_features_raw:
        cosine = np.dot(node_features_raw[u], node_features_raw[v]) / (feat_norms[u] * feat_norms[v])
    return np.array([deg_u, deg_v, cn, jacc, aa, pa, sc, cosine], dtype=np.float32)

print("Calcul features structurelles...")
train_struct = np.stack([structural_features(r.u, r.v) for r in train_df.itertuples()])
val_struct   = np.stack([structural_features(r.u, r.v) for r in val_df.itertuples()])
test_struct  = np.stack([structural_features(r.u, r.v) for r in test.itertuples()])

scaler       = StandardScaler().fit(train_struct)
train_struct = scaler.transform(train_struct).astype(np.float32)
val_struct   = scaler.transform(val_struct).astype(np.float32)
test_struct  = scaler.transform(test_struct).astype(np.float32)
struct_dim   = train_struct.shape[1]
print(f"Dim structurelle : {struct_dim}")

Calcul features structurelles...
Dim structurelle : 8


## 4. Graphe PyG — avec normalisation L2 des features

In [5]:
all_nodes   = sorted(G.nodes())
node_to_idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes   = len(all_nodes)

x_np = np.zeros((num_nodes, feature_dim), dtype=np.float32)
for node, idx in node_to_idx.items():
    if node in node_features_raw:
        x_np[idx] = node_features_raw[node]

# Normalisation L2 ligne par ligne : met toutes les features à la même échelle
# Important si les features sont du TF-IDF (valeurs très hétérogènes entre documents)
norms = np.linalg.norm(x_np, axis=1, keepdims=True) + 1e-9
x_np  = x_np / norms

x          = torch.tensor(x_np, dtype=torch.float)
edge_list  = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

data = Data(x=x, edge_index=edge_index, num_nodes=num_nodes)
print(data)

Data(x=[3597, 932], edge_index=[2, 8396], num_nodes=3597)


## 5. Tenseurs de paires

In [6]:
def pairs_to_tensors(df, struct_arr, node_to_idx):
    mask  = df["u"].isin(node_to_idx) & df["v"].isin(node_to_idx)
    idx   = np.where(mask.values)[0]
    valid = df.iloc[idx]
    return (
        torch.tensor([node_to_idx[u] for u in valid["u"]], dtype=torch.long),
        torch.tensor([node_to_idx[v] for v in valid["v"]], dtype=torch.long),
        torch.tensor(struct_arr[idx], dtype=torch.float),
        torch.tensor(valid["label"].values, dtype=torch.float),
    )

train_u, train_v, train_s, train_y = pairs_to_tensors(train_df, train_struct, node_to_idx)
val_u,   val_v,   val_s,   val_y   = pairs_to_tensors(val_df,   val_struct,   node_to_idx)
print(f"Paires — train : {len(train_y)} | val : {len(val_y)}")

Paires — train : 8396 | val : 2100


## 6. Modèles

### GNN : SAGEConv, 2 couches
2 couches = voisinage à 2 sauts, suffisant pour capturer la structure locale.
3 couches → oversmoothing (les embeddings convergent vers la moyenne globale).

### Décodeur symétrique
```
input = [z_u * z_v | |z_u - z_v| | struct_feats]
       = [produit élémentaire (similarité) | différence absolue (dissimilarité) | features de paire]
```
Invariant à l'ordre `(u,v)` vs `(v,u)` — cohérent avec un graphe non-orienté.

In [7]:
class GNN_model(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.4):
        super().__init__()
        # 2 couches SAGEConv : voisinage à 2 sauts
        self.conv1 = SAGEConv(input_size, hidden_size)
        self.bn1   = torch.nn.BatchNorm1d(hidden_size)
        self.conv2 = SAGEConv(hidden_size, output_size)
        self.bn2   = torch.nn.BatchNorm1d(output_size)
        self.dropout = dropout

    def forward(self, x, edge_index, drop_edge_p=0.0):
        # DropEdge : masque aléatoire d'arêtes pendant le training
        if drop_edge_p > 0:
            edge_index, _ = dropout_edge(edge_index, p=drop_edge_p, training=self.training)
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        return x


class LinkPredictor(torch.nn.Module):
    """
    Décodeur symétrique :
    input = [z_u * z_v | |z_u - z_v| | struct_feats]
    Invariant à l'ordre (u,v) ↔ (v,u).
    """
    def __init__(self, emb_dim, struct_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        in_dim = emb_dim * 2 + struct_dim  # prod + diff + struct
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z_u, z_v, struct_feats):
        prod = z_u * z_v               # similarité
        diff = torch.abs(z_u - z_v)    # dissimilarité
        x    = torch.cat([prod, diff, struct_feats], dim=-1)
        return self.net(x).squeeze(-1)

## 7. Entraînement avec DropEdge

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

HIDDEN_SIZE  = 256
EMB_SIZE     = 128
DROPOUT_GNN  = 0.4
DROPOUT_MLP  = 0.3
DROP_EDGE_P  = 0.4   # probabilité de masquer une arête à chaque forward
LR           = 1e-3
WEIGHT_DECAY = 5e-4
EPOCHS       = 600
PATIENCE     = 80

data    = data.to(device)
train_u, train_v, train_s, train_y = [t.to(device) for t in (train_u, train_v, train_s, train_y)]
val_u,   val_v,   val_s,   val_y   = [t.to(device) for t in (val_u,   val_v,   val_s,   val_y)]

gnn       = GNN_model(feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT_GNN).to(device)
predictor = LinkPredictor(EMB_SIZE, struct_dim, HIDDEN_SIZE, DROPOUT_MLP).to(device)
optimizer = torch.optim.Adam(
    list(gnn.parameters()) + list(predictor.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=30, verbose=True
)
pos_weight = torch.tensor([(train_y == 0).sum() / (train_y == 1).sum()]).to(device)


def evaluate(z, u, v, s, y):
    """Évaluation sans DropEdge (mode inférence)."""
    with torch.no_grad():
        logits = predictor(z[u], z[v], s)
        loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        auc    = roc_auc_score(y.cpu().numpy(), torch.sigmoid(logits).cpu().numpy())
    return loss.item(), auc


best_val_auc  = 0.0
best_state    = None
epochs_no_imp = 0

for epoch in range(1, EPOCHS + 1):
    gnn.train(); predictor.train()
    optimizer.zero_grad()

    # Forward avec DropEdge (masquage aléatoire d'arêtes)
    z      = gnn(data.x, data.edge_index, drop_edge_p=DROP_EDGE_P)
    logits = predictor(z[train_u], z[train_v], train_s)
    loss   = F.binary_cross_entropy_with_logits(logits, train_y, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        gnn.eval(); predictor.eval()
        # Inférence SANS DropEdge (toutes les arêtes utilisées)
        z_e             = gnn(data.x, data.edge_index, drop_edge_p=0.0)
        tr_loss, tr_auc = evaluate(z_e, train_u, train_v, train_s, train_y)
        va_loss, va_auc = evaluate(z_e, val_u,   val_v,   val_s,   val_y)
        scheduler.step(va_auc)
        print(f"Epoch {epoch:>3} | Train loss {tr_loss:.4f} AUC {tr_auc:.4f} | Val loss {va_loss:.4f} AUC {va_auc:.4f}")
        if va_auc > best_val_auc:
            best_val_auc  = va_auc
            epochs_no_imp = 0
            best_state    = {
                "gnn":       {k: v.cpu().clone() for k, v in gnn.state_dict().items()},
                "predictor": {k: v.cpu().clone() for k, v in predictor.state_dict().items()},
            }
        else:
            epochs_no_imp += 20
            if epochs_no_imp >= PATIENCE:
                print(f"Early stopping à l'epoch {epoch}.")
                break

print(f"\nMeilleur AUC validation : {best_val_auc:.4f}")

Device : cpu


d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch  20 | Train loss 1.2689 AUC 0.9569 | Val loss 1.3644 AUC 0.5475
Epoch  40 | Train loss 2.4922 AUC 0.9902 | Val loss 2.8308 AUC 0.5392
Epoch  60 | Train loss 1.5164 AUC 1.0000 | Val loss 2.8250 AUC 0.5478
Epoch  80 | Train loss 0.0583 AUC 1.0000 | Val loss 2.2302 AUC 0.5843
Epoch 100 | Train loss 0.0046 AUC 1.0000 | Val loss 2.1429 AUC 0.5936
Epoch 120 | Train loss 0.0011 AUC 1.0000 | Val loss 2.3632 AUC 0.5923
Epoch 140 | Train loss 0.0006 AUC 1.0000 | Val loss 2.5554 AUC 0.5821
Epoch 160 | Train loss 0.0005 AUC 1.0000 | Val loss 2.6659 AUC 0.5850
Epoch 180 | Train loss 0.0004 AUC 1.0000 | Val loss 2.7118 AUC 0.5808
Early stopping à l'epoch 180.

Meilleur AUC validation : 0.5936


## 8. Inférence — soumission en probas continues

In [9]:
gnn.load_state_dict({k: v.to(device) for k, v in best_state["gnn"].items()})
predictor.load_state_dict({k: v.to(device) for k, v in best_state["predictor"].items()})
gnn.eval(); predictor.eval()

# Inférence sans DropEdge
with torch.no_grad():
    z = gnn(data.x, data.edge_index, drop_edge_p=0.0)

test_s_tensor = torch.tensor(test_struct, dtype=torch.float, device=device)
u_list = [node_to_idx.get(int(r.u), -1) for r in test.itertuples()]
v_list = [node_to_idx.get(int(r.v), -1) for r in test.itertuples()]

scores = []
with torch.no_grad():
    for i, (ui, vi) in enumerate(zip(u_list, v_list)):
        if ui == -1 or vi == -1:
            scores.append(0.1)
        else:
            logit = predictor(
                z[ui].unsqueeze(0),
                z[vi].unsqueeze(0),
                test_s_tensor[i].unsqueeze(0)
            )
            scores.append(torch.sigmoid(logit).item())

test["score"] = scores
pd.DataFrame({"ID": range(len(scores)), "Predicted": scores}).to_csv("predictions.csv", index=False)

print(test[["u", "v", "score"]].head(10).to_string(index=False))
print(f"\nMin {min(scores):.4f} | Max {max(scores):.4f} | Mean {np.mean(scores):.4f}")

   u    v    score
3425 4524 0.989862
1620 2617 0.099390
4832 6317 0.998339
4984 7298 0.028413
 385 5481 0.536553
1722 2930 0.003603
1534 3330 0.999856
5015 6354 0.983653
 856 2504 0.010279
 851 5515 0.771844

Min 0.0000 | Max 1.0000 | Mean 0.5324
